### Algumas ideias:

**Cenário de problema: Operadora de internet/telefone quer reduzir cancelamentos.**

1. Objetivo:
**Prever cancelamento**.

Como a empresa pode agir antes:
*   Oferecer descontos
*   Melhorar atendimento
*   Criar campanhas de retenção

Exemplo de pergunta:
*   Qual perfil mais cancela?
*   Quanto a empresa perde?
*   Quais serviços ajudam na retenção?
*   Contrato mensal aumenta churn?
*   Qual faixa de valor tem maior evasão?





#Importações

In [246]:
#Importações
import pandas as pd
import sqlite3


#Análise

In [247]:
#Conexão com o repositório do github!rm -rf /content/hackathon-elas-tech-grupo7
!git clone https://github.com/joyexplorer/hackathon-elas-tech-grupo7.git

fatal: destination path 'hackathon-elas-tech-grupo7' already exists and is not an empty directory.


In [248]:
df = pd.read_csv('/content/hackathon-elas-tech-grupo7/dados/tratados/telco_customer_churn_tratado.csv')

In [249]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [250]:
# =====================================================
# ETAPA: Correção da coluna TotalCharges
# Converte valores textuais para numéricos.
# Valores inválidos ou vazios serão transformados em NaN.
# =====================================================

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [251]:
#Quantidade de linhas e colunas dos nossos dados
df.shape

(7032, 21)

In [252]:
df['TotalCharges'].isnull().sum()

np.int64(0)

In [253]:
# Remove registros com TotalCharges nulo
df = df.dropna(subset=['TotalCharges'])

In [254]:
df[df['TotalCharges'].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [255]:
# =====================================================
# SALVAMENTO DA BASE TRATADA
# Esta será a base oficial para SQL, dashboards
# e machine learning.
# =====================================================

df.to_csv('telco_customer_churn_tratado.csv', index=False)

from google.colab import files
files.download('telco_customer_churn_tratado.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [256]:
#São 21 colunas:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [257]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 


In [258]:
#Opção de renomear as colunas para português / Só se acharem necessário
#TotalCharges está como objeto e deveria ser numérico
#Talvez não precisamos de todas as colunas

'''df.rename(columns={
    'customerID': 'id_cliente',
    'gender': 'genero',
    'SeniorCitizen': 'idoso',
    'Partner': 'possui_parceiro',
    'Dependents': 'possui_dependentes',
    'tenure': 'tempo_contrato_meses',
    'PhoneService': 'servico_telefone',
    'MultipleLines': 'multiplas_linhas',
    'InternetService': 'servico_internet',
    'OnlineSecurity': 'seguranca_online',
    'OnlineBackup': 'backup_online',
    'DeviceProtection': 'protecao_dispositivo',
    'TechSupport': 'suporte_tecnico',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_filmes',
    'Contract': 'tipo_contrato',
    'PaperlessBilling': 'fatura_digital',
    'PaymentMethod': 'metodo_pagamento',
    'MonthlyCharges': 'valor_mensal',
    'TotalCharges': 'valor_total',
    'Churn': 'cancelamento'
}, inplace=True) '''



"df.rename(columns={\n    'customerID': 'id_cliente',\n    'gender': 'genero',\n    'SeniorCitizen': 'idoso',\n    'Partner': 'possui_parceiro',\n    'Dependents': 'possui_dependentes',\n    'tenure': 'tempo_contrato_meses',\n    'PhoneService': 'servico_telefone',\n    'MultipleLines': 'multiplas_linhas',\n    'InternetService': 'servico_internet',\n    'OnlineSecurity': 'seguranca_online',\n    'OnlineBackup': 'backup_online',\n    'DeviceProtection': 'protecao_dispositivo',\n    'TechSupport': 'suporte_tecnico',\n    'StreamingTV': 'streaming_tv',\n    'StreamingMovies': 'streaming_filmes',\n    'Contract': 'tipo_contrato',\n    'PaperlessBilling': 'fatura_digital',\n    'PaymentMethod': 'metodo_pagamento',\n    'MonthlyCharges': 'valor_mensal',\n    'TotalCharges': 'valor_total',\n    'Churn': 'cancelamento'\n}, inplace=True) "

#SQL

In [259]:
# =====================================================
# CONFIGURAÇÃO INICIAL: Conexão ao banco SQLite
# importar bibliotecas e criar a conexão principal (conn)
# com o banco de dados para execução das queries SQL.
# =====================================================
conn = sqlite3.connect('/content/hackathon-elas-tech-grupo7/sql/telco_churn_hackathon.db')

In [260]:
query = "SELECT * FROM clientes LIMIT 5;"
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [261]:
# =====================================================
# VERIFICAÇÃO INICIAL:
# Lista todas as tabelas disponíveis no banco de dados.
# e validar o nome exato da tabela antes das análises.
# =====================================================

query = """
SELECT name FROM sqlite_master WHERE type='table';
"""

pd.read_sql_query(query, conn)

,name
0,clientes


In [262]:
# =====================================================
# KPI 1: Total de clientes da base
# Esta consulta contabiliza o número total de registros
# (clientes únicos) presentes na base de dados.
# Serve como referência principal para cálculos
# percentuais futuros, como churn rate.
# =====================================================

query = """
SELECT COUNT(*) AS total_clientes
FROM clientes;
"""

total_clientes = pd.read_sql_query(query, conn)
total_clientes

,total_clientes
0,7032


In [263]:
# =====================================================
# KPI 2: Taxa geral de churn
# Calcula:
# - Número de clientes que cancelaram
# - Número de clientes ativos
# - Percentual de cada grupo
# mede a taxa de evasão da empresa.
# =====================================================

query = """
SELECT
    Churn,
    COUNT(*) AS total_clientes,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clientes), 2) AS percentual
FROM clientes
GROUP BY Churn;
"""

taxa_churn = pd.read_sql_query(query, conn)
taxa_churn

,Churn,total_clientes,percentual
0,No,5163,73.42
1,Yes,1869,26.58


In [264]:
# =====================================================
# KPI 3: Receita mensal total
# Soma todos os valores mensais pagos pelos clientes,
# representando o faturamento recorrente mensal atual.
#
# Importante para avaliação financeira global.
# =====================================================

query = """
SELECT
    ROUND(SUM(MonthlyCharges), 2) AS receita_mensal_total
FROM clientes;
"""

receita_total = pd.read_sql_query(query, conn)
receita_total

,receita_mensal_total
0,455661.0


In [265]:
# =====================================================
# KPI 4: Receita mensal em risco
# Soma o valor mensal associado aos clientes
# que já cancelaram o serviço.
#
# Essa métrica ajuda a estimar impacto financeiro
# da evasão de clientes.
# =====================================================

query = """
SELECT
    ROUND(SUM(MonthlyCharges), 2) AS receita_em_risco
FROM clientes
WHERE Churn = 'Yes';
"""

receita_risco = pd.read_sql_query(query, conn)
receita_risco

,receita_em_risco
0,139130.85


In [266]:
# =====================================================
# KPI 5: Ticket médio por churn
# Calcula o gasto médio mensal de clientes
# separados entre churn e não churn.
#
# Permite avaliar se clientes perdidos
# possuem maior valor monetário.
# =====================================================

query = """
SELECT
    Churn,
    ROUND(AVG(MonthlyCharges), 2) AS ticket_medio
FROM clientes
GROUP BY Churn;
"""

ticket_medio = pd.read_sql_query(query, conn)
ticket_medio

,Churn,ticket_medio
0,No,61.31
1,Yes,74.44


In [267]:
# =====================================================
# KPI 6: Churn por tipo de contrato
# Analisa distribuição de churn conforme:
# - Contrato mensal
# - Contrato anual
# - Contrato bienal
#
# Identifica contratos mais vulneráveis.
# =====================================================

query = """
SELECT
    Contract,
    Churn,
    COUNT(*) AS total
FROM clientes
GROUP BY Contract, Churn
ORDER BY Contract;
"""

churn_contrato = pd.read_sql_query(query, conn)
churn_contrato

,Contract,Churn,total
0,Month-to-month,No,2220
1,Month-to-month,Yes,1655
2,One year,No,1306
3,One year,Yes,166
4,Two year,No,1637
5,Two year,Yes,48


In [268]:
# =====================================================
# KPI 7: Churn por método de pagamento
# Avalia se certos métodos de pagamento
# possuem maior taxa de cancelamento.
#
# Pode revelar padrões financeiros
# e oportunidades de retenção.
# =====================================================

query = """
SELECT
    PaymentMethod,
    Churn,
    COUNT(*) AS total
FROM clientes
GROUP BY PaymentMethod, Churn;
"""

churn_pagamento = pd.read_sql_query(query, conn)
churn_pagamento

,PaymentMethod,Churn,total
0,Bank transfer (automatic),No,1284
1,Bank transfer (automatic),Yes,258
2,Credit card (automatic),No,1289
3,Credit card (automatic),Yes,232
4,Electronic check,No,1294
5,Electronic check,Yes,1071
6,Mailed check,No,1296
7,Mailed check,Yes,308


In [269]:
# =====================================================
# KPI 8: Tempo médio de permanência
# Mede quantos meses, em média,
# clientes churnados e não churnados
# permanecem na empresa.
#
# Fundamental para entender
# o ciclo de vida do cliente.
# =====================================================

query = """
SELECT
    Churn,
    ROUND(AVG(tenure), 2) AS media_tempo_contrato
FROM clientes
GROUP BY Churn;
"""

tempo_permanencia = pd.read_sql_query(query, conn)
tempo_permanencia

,Churn,media_tempo_contrato
0,No,37.65
1,Yes,17.98


In [270]:
# =====================================================
# KPI 9: Clientes de maior valor em risco
# Lista clientes churnados com maiores cobranças mensais.
#
# Ajuda a identificar clientes prioritários
# para estratégias de retenção premium.
# =====================================================

query = """
SELECT
    customerID,
    MonthlyCharges,
    TotalCharges,
    tenure
FROM clientes
WHERE Churn = 'Yes'
ORDER BY MonthlyCharges DESC
LIMIT 20;
"""

clientes_risco = pd.read_sql_query(query, conn)
clientes_risco

,customerID,MonthlyCharges,TotalCharges,tenure
0,8199-ZLLSA,118.35,7804.15,67
1,2889-FPWRM,117.80,8684.80,72
2,2302-ANTDP,117.45,5438.90,48
3,9053-JZFKV,116.20,7752.30,67
4,1444-VVSGW,115.65,7968.85,70
5,0201-OAMXR,115.55,8127.60,70
6,4361-BKAXE,114.50,4527.45,41
7,1555-DJEQW,114.20,7723.90,70
8,9158-VCTQB,113.60,4594.95,41
9,7279-BUYWN,113.20,4689.50,41


In [271]:
# =====================================================
# FINALIZAÇÃO:
# Fecha a conexão com o banco de dados
# =====================================================

conn.close()